# HW 6 - SQL - SOLUTIONS
## Part II - Databases and Python
## Name: 

For this part, you will make a SQLite database using Python and query it using SQL statements.  The database will be the Rent Ur Car Database.  Become familiar with the business by reading the background description on Canvas.

**1.  Create a Jupyter Notebook, import the sqllite3 and pandas libraries**

In [1]:
## SOLUTION ##

# import the sqlite3 library
import sqlite3

# import the pandas library
import pandas as pd

**2.  Create a sqlite3 database called car.db**

In [2]:
# create a connection object
connection = sqlite3.connect("car.db")

# create a cursor object
cursor = connection.cursor()

**3.	Read in Cars.csv, Customers.csv, Owners.csv, and Rentals.csv to create 4 tables for your sqlite database**

In [3]:
# In case of mistakes or for a fresh start, DROP tables
tables = ['Cars','Customers','Owners','Rentals']
#for table_name in tables:
#    cursor.execute(f'DROP TABLE {table_name}')

In [4]:
# SOLUTION
for table_name in tables:
    table = pd.read_csv(f'{table_name}.csv')
    table.to_sql(table_name, connection, if_exists='replace', index = False) 

**4. Verify you successfully imported the tables by checking the pandas dataframe heads for each**

In [5]:
from IPython.display import display  # allows the nice df display in jupyter from within loop

# SOLUTION
df_dict = dict() # diagnostic
for table_name in tables:
    sql_command = f'SELECT * from {table_name};'
    df = pd.read_sql_query(sql_command, connection)
    df_dict[table_name] = df
    print(20*'-',table_name,50*'-')
    display(df.head())

-------------------- Cars --------------------------------------------------


,Owner ID,Vehicle ID,Vehicle,Price Per Day
0,900,1,1992 Rover 216,169.0
1,909,10,2010 Mini Cooper Clubman,187.0
2,910,11,2008 Daewoo Kalos,129.0
3,911,12,2005 Land Rover Discovery,74.0
4,912,13,1992 Oldsmobile Eighty-Eight,181.0


-------------------- Customers --------------------------------------------------


,Customer ID,First Name,Last Name,Street Address,City,State,Zip Code,Email Address,Telephone Number,CC Type,CVV2,CC Expires
0,101,Ivy,Turk,3163 Jerry Dove Drive,Florence,SC,29501,IvyJTurk@armyspy.com,843-413-3824,Visa,723,10/1/2019
1,102,Rafael,Bishop,2978 James Street,Franklinville,NY,14737,RafaelMBishop@gustr.com,585-676-5429,MasterCard,15,6/1/2021
2,103,Daniel,Mattox,4413 Camden Place,Georgetown,SC,29440,DanielBMattox@superrito.com,843-904-8326,Visa,769,2/1/2021
3,104,Arthur,Baker,489 Christie Way,Ayer,MA,1432,ArthurMBaker@teleworm.us,978-772-6573,Visa,642,7/1/2019
4,105,Jennifer,Schrimsher,1335 Daffodil Lane,Mclean,VA,22101,JenniferASchrimsher@einrot.com,703-448-8919,MasterCard,324,6/1/2018


-------------------- Owners --------------------------------------------------


,Owner ID,First Name,Last Name,Street Address,City,State,Zip Code,Email Address,Telephone Number
0,901,Juan,Fishman,351 Shobe Lane,Fort Collins,CO,80524,JuanIFishman@fleckens.hu,970-494-7508
1,902,Grace,Job,398 Bagwell Avenue,Ocala,FL,34471,GraceJJob@dayrep.com,352-558-6887
2,903,Lee,Cope,2717 Vine Street,Morton Grove,IL,60053,LeeKCope@einrot.com,847-663-2840
3,904,Todd,Lewis,1892 Huntz Lane,Worcester,MA,1608,ToddMLewis@jourrapide.com,978-598-0790
4,905,Elizabeth,Costello,243 Sycamore Road,Bend,OR,97701,ElizabethTCostello@teleworm.us,541-388-2495


-------------------- Rentals --------------------------------------------------


,Rental ID,Vehicle ID,Customer ID,Start Date,End Date,Region
0,1,52,109,9/19/2017,9/21/2017,East
1,10,78,141,9/28/2017,10/1/2017,Midwest
2,11,14,133,9/29/2017,10/2/2017,Midwest
3,12,12,103,9/30/2017,10/2/2017,East
4,13,43,107,10/1/2017,10/5/2017,East


In [6]:
df_dict['Customers'].iloc[47]['Customer ID']

148

In [7]:
df_dict['Rentals'].iloc[68]['Customer ID']

148

**5:  Answer the following questions using the appropriate SQL statements.  Show the results of your query in a pandas dataframe.**

**a.  List the vehicles with the number of times it has been rented.  Show in descending order.**

In [8]:
# SOLUTION
sql_command = """SELECT Vehicle, Count(Rentals.[Vehicle ID])
FROM Cars
INNER JOIN Rentals
ON Rentals.[Vehicle ID] = Cars.[Vehicle ID]
GROUP BY Rentals.[Vehicle ID]
ORDER BY Count(Rentals.[Vehicle ID]) DESC
"""
df = pd.read_sql_query(sql_command, connection)
df.head(6)  # 6 not required, just to help show the full 5-way tie for 2nd

,Vehicle,Count(Rentals.[Vehicle ID])
0,2008 Volvo S80,4
1,2008 Maserati Coupe,3
2,2009 SsangYong Actyon,3
3,1996 Alfa Romeo Spider,3
4,1993 Mazda AZ-1,3
5,2002 Toyota Avalon,3


**b.  What is the average cost of a car per day?**

In [9]:
# SOLUTION
sql_command = """SELECT Avg([Price Per Day])
FROM Cars
"""
df = pd.read_sql_query(sql_command, connection)
df.head()

,Avg([Price Per Day])
0,162.591837


**c. Which owners have the priciest cars (>$250 per day cost)?  Show the owner’s first and last name and the price of the car.**

In [10]:
# SOLUTION
sql_command = """SELECT [First Name], [Last Name], [Price Per Day]
FROM Cars
INNER JOIN Owners
ON Owners.[Owner ID] = Cars.[Owner ID]
WHERE [Price Per Day] > 250;
"""
df = pd.read_sql_query(sql_command, connection)
df

,First Name,Last Name,Price Per Day
0,Millard,McElyea,256.0
1,Christopher,Perrin,289.0
2,Howard,Schramm,269.0
3,Wayne,Adair,454.0
4,Gary,Robbins,315.0
5,Stacey,Jones,370.0


**d. List the customer first and last name, along with the number of rentals they have done through Rent Ur Car.**

In [11]:
# SOLUTION
sql_command = """SELECT [First Name], [Last Name], Count(Rentals.[Vehicle ID])
FROM Customers
INNER JOIN Rentals
ON Rentals.[Customer ID] = Customers.[Customer ID]
GROUP BY [First Name], [Last Name]
HAVING Count(Rentals.[Vehicle ID]) < 2;
"""
df = pd.read_sql_query(sql_command, connection)
df

,First Name,Last Name,Count(Rentals.[Vehicle ID])
0,Alva,Velasquez,1
1,Antonio,Belanger,1
2,Audrey,Baker,1
3,Barry,Tuck,1
4,Bernice,Barton,1
5,Courtney,Kelley,1
6,Daniel,Mattox,1
7,Deborah,Childs,1
8,Ima,Carter,1
9,James,Fender,1


In [12]:
sql_command = """SELECT [First Name], [Last Name], Customers.[Customer ID], Count(Rentals.[Vehicle ID])
FROM Customers
INNER JOIN Rentals
ON Rentals.[Customer ID] = Customers.[Customer ID]
GROUP BY [First Name], [Last Name]
"""
df = pd.read_sql_query(sql_command, connection)
df

,First Name,Last Name,Customer ID,Count(Rentals.[Vehicle ID])
0,Alva,Velasquez,148,1
1,Angela,Smith,152,2
2,Antonio,Belanger,114,1
3,Arthur,Baker,104,2
4,Audrey,Baker,172,1
...,...,...,...,...
60,Wesley,Everson,116,2
61,William,Crawford,182,2
62,William,Nealon,112,1
63,William,Robinson,191,1


**e. The goal of this part is to create for each Region a new Rentals table that will go in the car database.  The new tables will be named: 'Rentals_East','Rentals_Midwest','Rentals_Mountain','Rentals_West'.**

i) Start by reading the Rentals table into a dataframe and display that dataframe.  You should have done this earlier in part 4, so can reuse that code here.

In [13]:
# re-loading the rentals table
rentals_df = pd.read_sql_query('SELECT * from Rentals;',connection)
rentals_df

,Rental ID,Vehicle ID,Customer ID,Start Date,End Date,Region
0,1,52,109,9/19/2017,9/21/2017,East
1,10,78,141,9/28/2017,10/1/2017,Midwest
2,11,14,133,9/29/2017,10/2/2017,Midwest
3,12,12,103,9/30/2017,10/2/2017,East
4,13,43,107,10/1/2017,10/5/2017,East
...,...,...,...,...,...,...
94,95,7,173,12/22/2017,12/25/2017,Mountain
95,96,4,149,12/23/2017,12/27/2017,Midwest
96,97,72,164,12/24/2017,12/25/2017,Mountain
97,98,78,193,12/25/2017,12/26/2017,West


ii) Next write and run code to create a list of the region names.  You may want to access the dataframe you created in the previous step.

In [14]:
regions = rentals_df['Region'].unique()
regions

array(['East', 'Midwest', 'Mountain', 'West'], dtype=object)

iii) Now loop through the regions, creating a sql query to get all records (and all fields) for each region using f strings, then writing the result to new database tables in car.db named Region_East, Region_Midwest, etc. You may want to use the pandas `to_sql` command.

In [15]:
for region in regions:
    sql_command = f"""SELECT *
                FROM Rentals 
                WHERE Region = '{region}'
                ;"""
    df = pd.read_sql_query(sql_command, connection)
    df.to_sql('Rentals_' + region, connection,if_exists='replace',index=False)
    print(df.head())

   Rental ID  Vehicle ID  Customer ID  Start Date    End Date Region
0          1          52          109   9/19/2017   9/21/2017   East
1         12          12          103   9/30/2017   10/2/2017   East
2         13          43          107   10/1/2017   10/5/2017   East
3         16          11          121   10/4/2017   10/6/2017   East
4         23          24          104  10/11/2017  10/13/2017   East
   Rental ID  Vehicle ID  Customer ID  Start Date    End Date   Region
0         10          78          141   9/28/2017   10/1/2017  Midwest
1         11          14          133   9/29/2017   10/2/2017  Midwest
2         14          53          128   10/2/2017   10/3/2017  Midwest
3         17          44          137   10/5/2017   10/7/2017  Midwest
4         29          12          142  10/17/2017  10/20/2017  Midwest
   Rental ID  Vehicle ID  Customer ID Start Date    End Date    Region
0         15          68          154  10/3/2017   10/6/2017  Mountain
1         18      

iv) Run the following command, interpret what it does, and comment on whether it suggests you have accomplished the previous task correctly or not.

In [16]:
# Run this command to display names of all the tables in your database
cursor.execute("""SELECT name
                from sqlite_master
                where type = 'table';""")
print(cursor.fetchall())

[('Cars',), ('Customers',), ('Owners',), ('Rentals',), ('Rentals_East',), ('Rentals_Midwest',), ('Rentals_Mountain',), ('Rentals_West',)]


v) Now check to see if the data was written to the tables by reading it back in from the tables.  Loop through the regions, creating sql queries via f strings to select all records from each table and display the results for each region.  Each record should appear only once in each region's table (or dataframe).

In [17]:
for region in regions:  
    sql_command = f"""SELECT *
                FROM {"Rentals_" + region}
                ;"""
    df = pd.read_sql_query(sql_command, connection)
    print(df)

    Rental ID  Vehicle ID  Customer ID  Start Date    End Date Region
0           1          52          109   9/19/2017   9/21/2017   East
1          12          12          103   9/30/2017   10/2/2017   East
2          13          43          107   10/1/2017   10/5/2017   East
3          16          11          121   10/4/2017   10/6/2017   East
4          23          24          104  10/11/2017  10/13/2017   East
5          24          53          117  10/12/2017  10/14/2017   East
6          28          38          101  10/16/2017  10/17/2017   East
7           3          24          106   9/21/2017   9/23/2017   East
8          33          38          117  10/21/2017  10/25/2017   East
9          34          88          114  10/22/2017  10/25/2017   East
10         41          25          105  10/29/2017  10/31/2017   East
11         45          51          101   11/2/2017   11/5/2017   East
12         46          75          124   11/3/2017   11/6/2017   East
13         49       

**Do not forget to close the connection when you are done.**

In [18]:
connection.close()